# Project 10: Visual Question Answering (VQA) across Different Question Types
### Fine-tuning `Salesforce/blip-vqa-base` on Kaggle GPU T4

> **Yêu cầu môi trường trên Kaggle:**
> - **Accelerator:** GPU T4 x 1
> - **Internet:** Enabled (Turn ON)

## 1. Cài đặt và Import các thư viện cần thiết

In [ ]:
!pip install -q transformers datasets pyarrow accelerate pillow tqdm pandas

import os, io, re, json, random, urllib.request
from collections import Counter
from PIL import Image
import pyarrow.parquet as pq
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import BlipProcessor, BlipForQuestionAnswering, get_cosine_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using Device:', device)
if torch.cuda.is_available():
    print('GPU Model:', torch.cuda.get_device_name(0))
    print('VRAM Available:', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')

## 2. Tải Dataset GQA Balanced Subsample (< 100 MB) và Phân loại 5 nhóm câu hỏi

In [ ]:
DATA_DIR = './data_gqa'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
os.makedirs(IMAGES_DIR, exist_ok=True)

INST_URL = 'https://huggingface.co/datasets/lmms-lab/GQA/resolve/main/testdev_balanced_instructions/testdev-00000-of-00001.parquet'
IMG_URL = 'https://huggingface.co/datasets/lmms-lab/GQA/resolve/main/testdev_balanced_images/testdev-00000-of-00001.parquet'

def download_file(url, out_path):
    if not os.path.exists(out_path):
        print(f'Downloading {os.path.basename(out_path)}...')
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resp, open(out_path, 'wb') as f:
            f.write(resp.read())
        print(f'Saved: {out_path}')

inst_parquet = os.path.join(DATA_DIR, 'instructions.parquet')
img_parquet = os.path.join(DATA_DIR, 'images.parquet')
download_file(INST_URL, inst_parquet)
download_file(IMG_URL, img_parquet)

# Trích xuất ảnh ra thư mục
print('Extracting images from parquet...')
img_table = pq.read_table(img_parquet)
img_ids = img_table['imageId'].to_pylist()
img_bytes = img_table['image'].to_pylist()

for img_id, b in tqdm(zip(img_ids, img_bytes), total=len(img_ids)):
    out_p = os.path.join(IMAGES_DIR, f'{img_id}.jpg')
    if not os.path.exists(out_p):
        raw = b['bytes'] if isinstance(b, dict) and 'bytes' in b else b
        Image.open(io.BytesIO(raw)).convert('RGB').save(out_p, format='JPEG', quality=95)

## 3. Quy tắc phân loại câu hỏi (Mapping to 5 Categories)

In [ ]:
def map_to_category(question: str, types: dict) -> str:
    sem = types.get('semantic', '')
    struct = types.get('structural', '')
    det = types.get('detailed', '').lower()
    ql = question.lower().strip()

    # 1. Counting
    if 'count' in det or ql.startswith('how many') or 'number of' in ql:
        return 'counting'
    # 2. Color
    if 'color' in det or 'what color' in ql or 'which color' in ql:
        return 'color'
    # 3. Spatial Relationships
    if sem == 'rel' or any(k in det for k in ['position', 'location']) or any(w in ql for w in ['left of', 'right of', 'behind', 'in front of', 'next to', 'above', 'below', 'under', 'where is', 'where are']):
        return 'spatial_relationships'
    # 4. Visual Reasoning
    if struct in ['compare', 'logical'] or any(k in det for k in ['compare', 'logic', 'same', 'different']) or ql.startswith('why'):
        return 'visual_reasoning'
    # 5. Object Recognition
    if sem in ['obj', 'cat'] or 'object' in det or 'category' in det or any(ql.startswith(w) for w in ['what is', 'what animal', 'what vehicle', 'what kind', 'is there a']):
        return 'object_recognition'
    return 'visual_reasoning'

inst_table = pq.read_table(inst_parquet)
pydict = inst_table.to_pydict()

samples = []
for i in range(len(pydict['id'])):
    q = pydict['question'][i]
    t = pydict['types'][i]
    samples.append({
        'id': pydict['id'][i],
        'image_id': pydict['imageId'][i],
        'image_path': os.path.join(IMAGES_DIR, f"{pydict['imageId'][i]}.jpg"),
        'question': q,
        'answer': str(pydict['answer'][i]).lower().strip(),
        'category': map_to_category(q, t)
    })

print('Dataset distribution across 5 categories:')
for k, v in Counter(s['category'] for s in samples).items():
    print(f'  - {k:<25}: {v:>5} ({v/len(samples)*100:.1f}%)')

## 4. Xây dựng PyTorch Dataset & DataLoader

In [ ]:
# Split theo image_id
random.seed(42)
all_imgs = sorted(list(set(s['image_id'] for s in samples)))
random.shuffle(all_imgs)

n_train = int(len(all_imgs) * 0.70)
n_val = int(len(all_imgs) * 0.15)

train_imgs = set(all_imgs[:n_train])
val_imgs = set(all_imgs[n_train:n_train + n_val])
test_imgs = set(all_imgs[n_train + n_val:])

train_samples = [s for s in samples if s['image_id'] in train_imgs]
val_samples = [s for s in samples if s['image_id'] in val_imgs]
test_samples = [s for s in samples if s['image_id'] in test_imgs]

print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')

class VQAKaggleDataset(Dataset):
    def __init__(self, data_list, processor=None):
        self.data = data_list
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item['image_path']).convert('RGB')
        question = item['question']
        answer = item['answer']

        if self.processor is not None:
            encoding = self.processor(images=image, text=question, padding='max_length', max_length=32, truncation=True, return_tensors='pt')
            encoding = {k: v.squeeze(0) for k, v in encoding.items()}
            labels = self.processor.tokenizer(answer, padding='max_length', max_length=16, truncation=True, return_tensors='pt').input_ids.squeeze(0)
            labels[labels == self.processor.tokenizer.pad_token_id] = -100
            encoding['labels'] = labels
            encoding['category'] = item['category']
            return encoding

        return {'image': image, 'question': question, 'answer': answer, 'category': item['category']}

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'input_ids': torch.stack([x['input_ids'] for x in batch]),
        'attention_mask': torch.stack([x['attention_mask'] for x in batch]),
        'labels': torch.stack([x['labels'] for x in batch]),
        'categories': [x['category'] for x in batch]
    }

## 5. Khởi tạo Mô hình `Salesforce/blip-vqa-base`

In [ ]:
MODEL_ID = 'Salesforce/blip-vqa-base'
processor = BlipProcessor.from_pretrained(MODEL_ID)
model = BlipForQuestionAnswering.from_pretrained(MODEL_ID).to(device)

train_ds = VQAKaggleDataset(train_samples, processor=processor)
val_ds = VQAKaggleDataset(val_samples, processor=processor)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=collate_fn, num_workers=2)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.05)
epochs = 3
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=len(train_loader)*epochs)
scaler = GradScaler()

## 6. Huấn luyện Mô hình với Mixed Precision (FP16)

In [ ]:
best_val_loss = float('inf')
SAVE_DIR = './best_blip_model'
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0
    pbar = tqdm(train_loader, desc=f'Train Epoch {epoch}')
    for batch in pbar:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            with autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values, labels=labels)
                total_val_loss += outputs.loss.item()

    avg_train = total_train_loss / len(train_loader)
    avg_val = total_val_loss / len(val_loader)
    print(f'Epoch {epoch}: Train Loss = {avg_train:.4f} | Val Loss = {avg_val:.4f}')

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        print(f'Saving best checkpoint to {SAVE_DIR}...')
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)

## 7. Đánh giá Benchmark trên 5 nhóm câu hỏi (Test Split)

In [ ]:
# Load best model
best_model = BlipForQuestionAnswering.from_pretrained(SAVE_DIR).to(device)
best_model.eval()

test_ds = VQAKaggleDataset(test_samples, processor=None)

def normalize(ans):
    ans = str(ans).lower().strip()
    ans = re.sub(r'[^\w\s]', '', ans)
    num_map = {'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4', 'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9', 'ten': '10'}
    return ' '.join([num_map.get(t, t) for t in ans.split()])

categories = ['object_recognition', 'color', 'counting', 'spatial_relationships', 'visual_reasoning']
stats = {c: {'correct': 0, 'total': 0} for c in categories}
total_correct = 0

print(f'Evaluating {len(test_ds)} test samples...')
with torch.no_grad():
    for i in tqdm(range(len(test_ds))):
        item = test_ds[i]
        inputs = processor(images=item['image'], text=item['question'], return_tensors='pt').to(device)
        out = best_model.generate(**inputs, max_new_tokens=10)
        pred = processor.decode(out[0], skip_special_tokens=True).strip()

        cat = item['category']
        is_correct = (normalize(pred) == normalize(item['answer']))

        stats[cat]['total'] += 1
        if is_correct:
            stats[cat]['correct'] += 1
            total_correct += 1

print('\n' + '='*65)
print(f"{'Question Type':<25} | {'Correct':<8} | {'Total':<8} | {'Accuracy (%)':<12}")
print('-'*65)
for c in categories:
    cor = stats[c]['correct']
    tot = stats[c]['total']
    acc = (cor / tot * 100) if tot > 0 else 0.0
    print(f"{c:<25} | {cor:<8} | {tot:<8} | {acc:<12.2f}%")
print('-'*65)
overall_acc = (total_correct / len(test_ds) * 100) if len(test_ds) > 0 else 0.0
print(f"{'OVERALL':<25} | {total_correct:<8} | {len(test_ds):<8} | {overall_acc:<12.2f}%")
print('='*65)